In [123]:
import pandas as pd
import numpy as np
from scipy import stats
import math
import collections

# Load the merged data
df = pd.read_csv("numbers_all.csv")
numbers = df["random_number"].to_numpy()

# Convert integers to a flat bit string 
# Each number is 16 bits wide (0–65535), zero-pad to 16 characters
# All 4880 numbers become one long string of 0s and 1s: 4880 × 16 = 78,080 bits
# Most NIST tests work on raw bit streams, not the integers themselves
# should be around 0.5

bits = np.array([int(b) for n in numbers for b in format(int(n), '016b')])

print(f"Numbers: {len(numbers)}")
print(f"Bits: {len(bits)}")
print(f"Ones: {bits.sum()}, Zeros: {len(bits) - bits.sum()}")
print(f"Proportion of ones: {bits.mean():.4f}")

Numbers: 9870
Bits: 157920
Ones: 79098, Zeros: 78822
Proportion of ones: 0.5009


In [124]:
# TEST 1: Monobit (Frequency) Test
# The simplest NIST test. Asks if there are roughly equal 0s and 1s?
# We map bits to +1/-1 and sum them. If truly random, the sum should be near 0.
# The p-value tells us how likely this imbalance is by chance.
# p >= 0.05 = PASS (imbalance is not statistically significant)

n = len(bits)
s = np.sum(2 * bits - 1)
s_obs = abs(s) / math.sqrt(n)
p_val = math.erfc(s_obs / math.sqrt(2))

print(f"Ones: {bits.sum()}, Zeros: {n - bits.sum()}")
print(f"S_obs: {s_obs:.6f}")
print(f"p-value: {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

Ones: 79098, Zeros: 78822
S_obs: 0.694529
p-value: 0.487350
Result: PASS


In [125]:
# TEST 2: Block Frequency Test
# Same idea as monobit but applied block-by-block instead of the whole sequence
# Chop the bit stream into blocks of 128 bits and check that each block has ~50% ones
# A global balance can hide local biases
# chi-squared statistic across all blocks; p >= 0.05 = PASS

M = 128 # block size (NIST recommends M >= 20)
N_blks = n // M # number of complete blocks, discard leftover

chi_sq = 0.0
for i in range(N_blks):
    block = bits[i*M : (i+1)*M]
    pi_i = np.sum(block) / M # proportion of 1s in this block
    chi_sq += (pi_i - 0.5) ** 2

chi_sq *= 4 * M
p_val = 1 - stats.chi2.cdf(chi_sq, df=N_blks)

print(f"Block size: {M}, Number of blocks: {N_blks}")
print(f"Chi-squared: {chi_sq:.6f}")
print(f"p-value: {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

Block size: 128, Number of blocks: 1233
Chi-squared: 1211.843750
p-value: 0.660947
Result: PASS


In [126]:
# TEST 3: Runs Test
# A run is an unbroken sequence of identical bits (000 11111)
# This test checks whether runs of 0s and 1s alternate at the expected rate
# Too few runs = bits are "sticky" (clustering) 
# Too many = alternating too regularly
# Before running, we check the proportion π is close enough to 0.5 


pi = bits.sum() / n # proportion of 1s

# pre-condition check
if abs(pi - 0.5) >= (2 / math.sqrt(n)):
    print("Pre-condition FAILED")
else:
    # count runs: a new run starts every time a bit differs from the previous one
    V_n = 1 + np.sum(bits[:-1] != bits[1:])
    
    numerator   = abs(V_n - 2 * n * pi * (1 - pi))
    denominator = 2 * math.sqrt(2 * n) * pi * (1 - pi)
    p_val = math.erfc(numerator / denominator)

    print(f"Proportion of ones (π): {pi:.6f}")
    print(f"Number of runs (V_n): {V_n}")
    print(f"p-value: {p_val:.6f}")
    print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

Proportion of ones (π): 0.500874
Number of runs (V_n): 78927
p-value: 0.869047
Result: PASS


In [127]:
# TEST 4: Longest Run of Ones in a Block
# For each block of 128 bits, we find the longest consecutive streak of 1s
# We then compare the distribution of those streak lengths to what NIST expects for a truly random sequence
# This catches "burst" behavior a broken RNG might produce abnormally
# long or short streaks of 1s even if the overall count looks fine.
# NIST defines 6 bins for M=128: ≤4, 5, 6, 7, 8, ≥9

M = 128
N_blks = n // M

# NIST SP 800-22 Table 2 expected probabilities for block size M=128
pi_expected = [0.1174, 0.2430, 0.2493, 0.1752, 0.1027, 0.1124]

def longest_run_of_ones(block):
    max_run, cur_run = 0, 0
    for b in block:
        if b == 1:
            cur_run += 1
            max_run = max(max_run, cur_run)
        else:
            cur_run = 0
    return max_run

def bin_run(r):
    # map run length into one of NIST's 6 bins
    if r <= 4:   return 0
    elif r == 5: return 1
    elif r == 6: return 2
    elif r == 7: return 3
    elif r == 8: return 4
    else:        return 5

observed = [0] * 6
for i in range(N_blks):
    block = bits[i*M : (i+1)*M]
    observed[bin_run(longest_run_of_ones(block))] += 1

chi_sq = sum(
    (observed[j] - N_blks * pi_expected[j]) ** 2 / (N_blks * pi_expected[j])
    for j in range(6)
)
p_val = 1 - stats.chi2.cdf(chi_sq, df=5)

print(f"Observed bin counts (≤4, 5, 6, 7, 8, ≥9): {observed}")
print(f"Expected bin counts: {[round(N_blks * p, 1) for p in pi_expected]}")
print(f"Chi-squared: {chi_sq:.6f}")
print(f"p-value: {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

Observed bin counts (≤4, 5, 6, 7, 8, ≥9): [158, 290, 323, 199, 132, 131]
Expected bin counts: [144.8, 299.6, 307.4, 216.0, 126.6, 138.6]
Chi-squared: 4.298531
p-value: 0.507283
Result: PASS


In [128]:
# TEST 5: Chi-Squared Uniformity Test
# This one works on the raw integers, not the bit stream
# We divide the range 0–65535 into 256 equal buckets and count how many numbers fall in each
# If the generator is uniform, every bucket should have roughly the same count
# This is not an official NIST SP 800-22 test but is a standard sanity
# check for QRNG output and easy to explain in the paper

n_bins = 256
observed, bin_edges = np.histogram(numbers, bins=n_bins, range=(0, 65536))
chi_sq, p_val = stats.chisquare(observed)

print(f"Numbers: {len(numbers)}, Bins: {n_bins}")
print(f"Expected per bin: {len(numbers)/n_bins:.1f}")
print(f"Observed min/max per bin: {observed.min()} / {observed.max()}")
print(f"Chi-squared: {chi_sq:.6f}")
print(f"p-value: {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

Numbers: 9870, Bins: 256
Expected per bin: 38.6
Observed min/max per bin: 20 / 57
Chi-squared: 251.155826
p-value: 0.556271
Result: PASS


In [129]:
# TEST 6: Kolmogorov-Smirnov Uniformity Test
# Like chi-squared, this checks that our integers are uniformly distributed, but it works differently
# it compares the empirical CDF of our data to a perfect uniform CDF
# The KS statistic D is the maximum gap between the two curves at any point
# It's more sensitive than chi-squared at detecting subtle drift in the tails
# We normalize numbers to [0,1] first so we can compare to Uniform(0,1)

normalized = numbers / 65536.0 # map 0–65535 → 0–1

ks_stat, p_val = stats.kstest(normalized, "uniform")

print(f"KS statistic (D): {ks_stat:.6f}")
print(f"p-value: {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

KS statistic (D): 0.009924
p-value: 0.283570
Result: PASS


In [130]:
# TEST 7: Serial Test
# Checks that all 2-bit pairs appear equally often 00 01 10 11
# A truly random sequence should produce each pair ~25% of the time
# This catches short-range correlations between adjacent bits
# a generator that tends to repeat the same bit would show too many 00 and 11
# We use overlapping windows (each bit starts a new pair) with wrap-around

import collections

m = 2

def count_ngrams(b, size):
    counts = collections.Counter()
    extended = np.append(b, b[:size-1]) # wrap-around so last bits pair with first
    for i in range(len(b)):
        gram = tuple(extended[i : i+size])
        counts[gram] += 1
    return counts

def psi_sq(b, size):
    if size == 0:
        return 0.0
    counts = count_ngrams(b, size)
    total  = 2 ** size
    return (total / len(b)) * sum(c**2 for c in counts.values()) - len(b)

psi2 = psi_sq(bits, m)
psi1 = psi_sq(bits, m - 1)
psi0 = psi_sq(bits, m - 2)

delta1 = psi2 - psi1
delta2 = psi2 - 2*psi1 + psi0

p1 = 1 - stats.chi2.cdf(delta1, df=2**(m-1))
p2 = 1 - stats.chi2.cdf(delta2, df=2**(m-2))

# report the more conservative of the two p-values
p_val = min(p1, p2)

pair_counts = count_ngrams(bits, 2)
print("2-bit pair counts:")
for pair in [(0,0),(0,1),(1,0),(1,1)]:
    print(f"  {pair}: {pair_counts[pair]} (expected {n//4})")
print(f"p-value: {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

2-bit pair counts:
  (0, 0): 39359 (expected 39480)
  (0, 1): 39463 (expected 39480)
  (1, 0): 39463 (expected 39480)
  (1, 1): 39635 (expected 39480)
p-value: 0.774277
Result: PASS


In [131]:
# TEST 8: Approximate Entropy Test
# Measures how complex/predictable the sequence is
# It compares the frequency of overlapping m-bit blocks to (m+1)-bit blocks
# The idea: if knowing m bits lets you predict the next bit, entropy is low
# High ApEn = high complexity = good randomness
# Theoretical max for a perfect RNG is ln(2) ≈ 0.693

m = 3 # block size — NIST recommends m such that m < log2(n) - 5

def phi(block_len):
    # compute the average log probability of all overlapping blocks of this length
    counts = collections.Counter()
    extended = np.append(bits, bits[:block_len - 1]) # wrap-around
    for i in range(n):
        gram = tuple(extended[i : i+block_len])
        counts[gram] += 1
    total = sum(counts.values())
    return sum((c/total) * math.log(c/total) for c in counts.values())

phi_m  = phi(m)
phi_m1 = phi(m + 1)
ap_en  = phi_m - phi_m1

chi_sq = 2 * n * (math.log(2) - ap_en)
p_val  = 1 - stats.chi2.cdf(chi_sq, df=2**m)

print(f"Approximate Entropy (ApEn): {ap_en:.6f}")
print(f"Theoretical max (ln 2):     {math.log(2):.6f}")
print(f"Chi-squared: {chi_sq:.6f}")
print(f"p-value: {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

Approximate Entropy (ApEn): 0.693135
Theoretical max (ln 2):     0.693147
Chi-squared: 3.905024
p-value: 0.865589
Result: PASS


In [132]:
# TEST 9: Cumulative Sums (Cusums)
# We do a random walk through the bit stream: +1 for each 1 bit, -1 for each 0
# The test checks that the maximum excursion is consistent with what a truly random sequence would produce
# A biased generator would cause the walk to drift steadily in one direction
# We run it twice — forward through the sequence and backward
# Both directions should pass

from scipy.stats import norm

def cusums_test(b, forward=True):
    seq = 2 * b - 1 # convert to +/-1
    if not forward:
        seq = seq[::-1]
    
    walk = np.cumsum(seq)
    z = np.max(np.abs(walk)) # maximum absolute excursion
    n = len(b)
    
    # p-value formula from NIST SP 800-22 Section 2.13
    # it's a sum of normal CDF terms
    def p_value(z, n):
        sum1 = sum(
            norm.cdf((4*k+1)*z / math.sqrt(n)) - norm.cdf((4*k-1)*z / math.sqrt(n))
            for k in range(int((-n/z + 1)/4), int((n/z - 1)/4) + 1)
        )
        sum2 = sum(
            norm.cdf((4*k+3)*z / math.sqrt(n)) - norm.cdf((4*k+1)*z / math.sqrt(n))
            for k in range(int((-n/z - 3)/4), int((n/z - 1)/4) + 1)
        )
        return 1 - sum1 + sum2

    p = p_value(z, n)
    return z, p

z_fwd, p_fwd = cusums_test(bits, forward=True)
z_bwd, p_bwd = cusums_test(bits, forward=False)

print(f"Forward  — max excursion: {z_fwd}, p-value: {p_fwd:.6f}, Result: {'PASS' if p_fwd >= 0.05 else 'FAIL'}")
print(f"Backward — max excursion: {z_bwd}, p-value: {p_bwd:.6f}, Result: {'PASS' if p_bwd >= 0.05 else 'FAIL'}")

Forward  — max excursion: 445, p-value: 0.524034, Result: PASS
Backward — max excursion: 358, p-value: 0.721564, Result: PASS


In [133]:
# TEST 10: Discrete Fourier Transform (Spectral) Test
# Converts the bit stream into the frequency domain using FFT
# A truly random sequence should have no repeating patterns
# Repeating patterns would show up as spikes in the frequency spectrum
# We count how many frequency components exceed a threshold and compare to what we'd expect from a random sequence (~95% should fall below it)

from numpy.fft import fft

# convert bits to +1/-1
seq = 2 * bits - 1

# apply FFT and take magnitude of first half (spectrum is symmetric)
frequencies = fft(seq)
magnitudes  = np.abs(frequencies[:len(seq) // 2])

# NIST threshold: 95% of peaks should fall below this value
T  = math.sqrt(math.log(1 / 0.05) * n)

# count how many peaks are below the threshold
N0 = 0.95 * (n / 2) # expected count below threshold
N1 = np.sum(magnitudes < T) # observed count below threshold

d = (N1 - N0) / math.sqrt(n * 0.95 * 0.05 / 4)
p_val  = math.erfc(abs(d) / math.sqrt(2))

print(f"Threshold T:         {T:.4f}")
print(f"Expected below T:    {N0:.1f}")
print(f"Observed below T:    {N1}")
print(f"d statistic:         {d:.6f}")
print(f"p-value:             {p_val:.6f}")
print(f"Result: {'PASS' if p_val >= 0.05 else 'FAIL'}")

Threshold T:         687.8125
Expected below T:    75012.0
Observed below T:    75039
d statistic:         0.623488
p-value:             0.532964
Result: PASS


In [134]:
!pip install nistrng

In [136]:
from nistrng import run_all_battery, SP800_22R1A_BATTERY
import numpy as np

bits_nistrng = bits.astype(np.int8)

results = run_all_battery(bits_nistrng, SP800_22R1A_BATTERY)

print(f"{'Test':<40} {'Score':>10} {'Result':>20}")
print("─" * 80)
for item in results:
    if item is None:
        continue
    result, params = item
    score = result.score if result.score is not None else "N/A"
    passed = "PASS" if result.passed else "FAIL"
    print(f"{result.name:<40} {str(score):>10} {passed:>10}")

Test                                          Score               Result
────────────────────────────────────────────────────────────────────────────────
Monobit                                  0.48735039033532057       PASS
Frequency Within Block                   0.7626137191431287       PASS
Runs                                     0.8690465165296691       PASS
Longest Run Ones In A Block              0.5858365317433142       PASS
Binary Matrix Rank                       0.10097522643935311       PASS
Discrete Fourier Transform                      0.0       FAIL
Non Overlapping Template Matching        0.307220238739824       PASS
Serial                                          0.0       FAIL
Approximate Entropy                             0.0       FAIL
Cumulative Sums                                 0.0       FAIL
Random Excursion                         0.68314332909863       FAIL
Random Excursion Variant                 0.17025213718165474       PASS
